# OpenCabinet Diffusion Policy -- Colab Training

Train the diffusion UNet low-dim policy on a Colab GPU.

**Before running this notebook:**
1. On your local machine, find the augmented parquet data (printed by `05b_augment_handle_data.py`)
2. Zip it: `cd <dataset_path> && zip -r augmented_data.zip augmented/`
3. Upload `augmented_data.zip` to your Google Drive root (or any folder -- adjust the path below)
4. Make sure your latest code is pushed to GitHub

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repo + Diffusion Policy

In [ ]:
import os

REPO_URL = "https://github.com/zlabute/cs-188-Final-Project-Kitchen-Cabinet.git"
DP_URL = "https://github.com/robocasa-benchmark/diffusion_policy.git"
PROJECT_DIR = "/content/cs188-cabinet-door-project"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

DP_DIR = os.path.join(PROJECT_DIR, "cabinet_door_project", "diffusion_policy")
if not os.path.exists(DP_DIR):
    !git clone {DP_URL} {DP_DIR}

## 3. Install Dependencies

In [ ]:
!pip install -e {DP_DIR}
!pip install diffusers pyarrow pyyaml tqdm

## 4. Prepare Augmented Data

Unzip the augmented parquet files you uploaded to Google Drive.

Edit `DRIVE_ZIP_PATH` below if you placed `augmented_data.zip` somewhere other than `My Drive/`.

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/augmented_data.zip"
DATA_DIR = "/content/augmented_data"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q {DRIVE_ZIP_PATH} -d {DATA_DIR}

aug_dir = os.path.join(DATA_DIR, "augmented")
parquet_files = [f for f in os.listdir(aug_dir) if f.endswith(".parquet")]
print(f"Found {len(parquet_files)} augmented parquet files in {aug_dir}")

## 5. Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected -- go to Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 6. Train

Checkpoints are saved directly to Google Drive so they survive runtime disconnects.

Adjust `--epochs` and `--batch_size` as needed. With a Colab T4 (16 GB VRAM), `batch_size=128` should work fine.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/cabinet_checkpoints"

!cd {PROJECT_DIR}/cabinet_door_project && python 09_train_lowdim_unet.py \
    --dataset_path {DATA_DIR} \
    --checkpoint_dir {CKPT_DIR} \
    --epochs 100 \
    --batch_size 128

## 7. Retrieve Checkpoints

Checkpoints are already on Google Drive at `My Drive/cabinet_checkpoints/`.

On your **local machine**, download `best_diffusion_policy.pt` (or `final_diffusion_policy.pt`) and place it in:
```
cabinet_door_project/tmp/cabinet_policy_checkpoints/
```

Then evaluate locally:
```bash
cd cabinet_door_project
python 07_evaluate_policy.py --checkpoint tmp/cabinet_policy_checkpoints/best_diffusion_policy.pt
python 08_visualize_policy_rollout.py --checkpoint tmp/cabinet_policy_checkpoints/best_diffusion_policy.pt
```

In [ ]:
import glob

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt")))
print("Saved checkpoints on Google Drive:")
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c):>40s}  ({size_mb:.1f} MB)")